In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay
)

In [2]:
master = pd.read_csv(
    "../data/processed/se3_master_dataset.csv",
    parse_dates=["timestamp"]
)

labels = pd.read_csv(
    "../data/processed/se3_anomaly_labels.csv",
    parse_dates=["timestamp"]
)

In [3]:
labels["high_price_anomaly"] = (
    labels["high_price_anomaly"]
    .astype(str)
    .map({"True": 1, "False": 0})
)

In [4]:
labels["high_price_anomaly"].value_counts()

high_price_anomaly
0    17790
1      567
Name: count, dtype: int64

In [6]:
data = master.merge(
    labels[["timestamp", "high_price_anomaly"]],
    on="timestamp",
    how="left"
)

In [7]:
data.shape

(18816, 23)

In [8]:
feature_cols = [
    "price_24h",
    "price_48h",
    "price_168h",

    "load_forecast_mw",

    "temp_stockholm",
    "wind_stockholm",
    "temp_gothenburg",
    "wind_gothenburg",
    "temp_orebro",
    "wind_orebro",

    "hour",
    "weekday",
    "month",
    "is_weekend"
]

In [9]:
data[
    feature_cols + ["high_price_anomaly"]
].isna().sum()

price_24h             482
price_48h             505
price_168h            616
load_forecast_mw       24
temp_stockholm          0
wind_stockholm          0
temp_gothenburg         0
wind_gothenburg         0
temp_orebro             0
wind_orebro             0
hour                    0
weekday                 0
month                   0
is_weekend              0
high_price_anomaly    459
dtype: int64

In [10]:
model_data = data.dropna(
    subset=feature_cols + ["high_price_anomaly"]
).copy()

In [11]:
model_data.groupby("split").size()

split
test          7792
train         8240
validation    1284
dtype: int64

In [12]:
model_data.groupby("split")["high_price_anomaly"].agg(
    total="size",
    anomalies="sum"
)

,total,anomalies
split,,
test,7792,337.0
train,8240,73.0
validation,1284,119.0
